# MACURA on Drones — a paper-vs-project study notebook

**MACURA in one sentence:** a Dyna-style model-based RL method that makes each
imagined rollout *as long as the model can be trusted* — it continues while the
dynamics ensemble agrees with itself and truncates the instant disagreement
(epistemic uncertainty) rises (Frauenknecht et al., *Trust the Model Where It
Trusts Itself*, ICML 2024, [arXiv:2405.19014](https://arxiv.org/abs/2405.19014)).

**This project** transfers that idea from the paper's stable MuJoCo locomotion
tasks to an **unstable Skydio X2 quadrotor** (hover / stabilization), where model
error compounds fast — the regime where adaptive truncation should matter most.
We run the **four algorithms the paper studies** — MACURA, MBPO, M2AC, SAC — with
a shared SAC backbone and a shared ensemble, so any difference is attributable to
the **rollout strategy alone**.

**What you will see:** (1) the paper's ideas explained with small illustrative
plots; (2) an environment study showing why a drone is a hard test; (3) a
side-by-side of every design choice (paper vs this project); (4) the four-way
training comparison; (5) the result figures and flight videos; (6) an honest
discussion and reproducibility footer.

## 0.1 How to read this notebook

Each code cell is preceded by prose explaining *why it exists* and *what to look
for*. The notebook is a guide, not just a script.

**Paper → project file mapping** (every concept lives in a pure-function module;
the notebook only orchestrates and plots):

| Paper concept | Equation / section | This repo |
|---|---|---|
| Probabilistic ensemble | Sec 2.2, Eq 3–5 | `models/ensemble.py` |
| GJS uncertainty | Eq 15–19, Sec 5.2 | `compute_gjs` in `algorithms/macura.py` |
| Adaptive threshold κ | Eq 21, Sec 6.1 | `update_kappa`, `macura_rollout` |
| UTD gradient scaling | Eq 22, Sec 6.1 | `gradient_steps` in `algorithms/macura.py` |
| MBPO fixed schedule | Janner 2019 | `algorithms/mbpo.py` |
| M2AC masking | Pan 2020 | `algorithms/m2ac.py` |
| SAC backbone | Haarnoja 2018 | `algorithms/sac.py` (Stable-Baselines3) |
| Drone task / reward | (new) | `envs/drone_env.py` |
| Training (Dyna loop) | Fig 1 | `training/train.py` |
| Figures | Figs 4–8 | `viz/plots.py` |

**`SMOKE` flag:** when `True` (default) the notebook runs tiny step counts so it
executes top-to-bottom in minutes for review. Set `False` for the full matrix.

**Outputs:** every figure and video is saved under `{DRIVE}/plots` and
`{DRIVE}/videos` on Google Drive *and* displayed inline.

## 0.2 Mount Google Drive

All artifacts mirror to Drive so they survive a Colab session reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0.3 Assert a GPU is present (hard requirement)

All training MUST run on the Colab GPU. This cell stops the notebook with a clear
message if no CUDA device is available (set **Runtime → Change runtime type →
GPU**). We also set `MUJOCO_GL=egl` for headless rendering on the GPU.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'   # headless MuJoCo rendering on the Colab GPU
import torch
assert torch.cuda.is_available(), (
    'No CUDA GPU found. Set Runtime -> Change runtime type -> GPU and rerun.')
DEVICE = 'cuda'
print('GPU OK:', torch.cuda.get_device_name(0))

## 0.4 GitHub token (private repo) — securely

The project repo is private. Store a PAT (`repo` scope) in Colab **Secrets** as
`GITHUB_TOKEN` (recommended) or enter it at the hidden prompt. It is held only in
memory for the clone and never written to disk.

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

## 0.5 Bootstrap: clone + Skydio X2 model + install

Fetches `setup_colab.sh` from the repo and runs it: clones this repo +
`mujoco_menagerie` (Skydio X2), installs `requirements.txt` (Stable-Baselines3 +
MuJoCo + torch), and creates the Drive folders.

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=macura-drone
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')

In [ ]:
# --- Runtime hardening ---
# Remove the unmaintained legacy 'gym' (incompatible with NumPy 2.0; can cause
# native issues alongside numpy 2). We use gymnasium throughout.
!pip uninstall -y gym >/dev/null 2>&1 || true
import os
# 'egl' lets `import mujoco` succeed on Colab GPUs. Creating a MuJoCo *Renderer*
# can still hard-crash the kernel on some GPUs (e.g. Tesla T4), so rendering is
# OPT-IN via RENDER and is NOT required for any result — the env study, training,
# and all plots run without it.
os.environ['MUJOCO_GL'] = 'egl'
RENDER = False   # set True ONLY to attempt still-frame / video rendering
print('hardening done: gym removed; MUJOCO_GL=egl; RENDER =', RENDER)

## 0.6 Load the config + set the `SMOKE` flag

`configs/macura_drone.yaml` is the single source of truth. `make_cfg` returns a
copy with tiny step counts when `SMOKE=True` (fast review) — no hyperparameters
are hard-coded in the notebook.

In [ ]:
import copy, yaml
with open('/content/macura-drone/configs/macura_drone.yaml') as f:
    CFG = yaml.safe_load(f)
CFG['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
DRIVE = CFG['experiment']['drive_root']

SMOKE = True   # <-- set False for the full matrix (long)

def make_cfg(smoke):
    c = copy.deepcopy(CFG)
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=250, eval_episodes=2, seeds=[0])
        c['rollout']['freq_steps'] = 250
        c['rollout']['num_rollouts'] = 100
    return c

cfg_run = make_cfg(SMOKE)
print('SMOKE =', SMOKE, '| steps =', cfg_run['experiment']['total_env_steps'],
      '| seeds =', cfg_run['experiment']['seeds'])

---
# Part 1 — What the paper introduces

This part is a stand-alone study guide: mostly prose and small *illustrative*
plots (clearly labelled synthetic where noted). It explains the ideas before any
training runs.

## 1.1 The problem: model exploitation

Model-based RL is sample-efficient because the agent trains on *imagined* rollouts
from a learned dynamics model instead of expensive real interaction. The danger
(paper Sec. 1): a learned model has small errors that **compound** as you imagine
further ahead, so a policy trained on long fantasy rollouts learns to exploit
dynamics that don't exist. MBPO (Janner 2019) increases rollout length on a fixed
schedule; MACURA instead asks *where* (not *when*) to trust the model.

The plot below is **synthetic/illustrative**: it sketches how error grows with the
rollout horizon, and where uncertainty-adaptive truncation would cut it off.

In [ ]:
from viz import plots
plots.plot_model_error_growth(save_path=f'{DRIVE}/plots/p1_error_growth.png');

## 1.2 The four algorithms

All four share the **same** SAC backbone and (for model-based ones) the **same**
ensemble — only the rollout strategy differs (this is what makes the comparison
fair).

| Algorithm | Type | Rollout strategy | Uncertainty use | Role | Repo |
|---|---|---|---|---|---|
| **MACURA** | Model-based | **Adaptive** truncation (κ vs GJS) | drives adaptive κ | proposed method | `algorithms/macura.py` (`macura_rollout`) |
| **MBPO** | Model-based | **Fixed** truncated-linear schedule | none | no-adaptivity baseline | `algorithms/mbpo.py` (`mbpo_rollout`) |
| **M2AC** | Model-based | fixed length, **mask** worst steps | filter, not length | alt. uncertainty use | `algorithms/m2ac.py` (`m2ac_rollout`) |
| **SAC** | Model-free | **no** model rollouts | — | reference baseline | `algorithms/sac.py` (`build_sac`) |

The headline contrasts: **MACURA vs SAC** (does the model help?), **MACURA vs
MBPO** (does adaptivity matter?), **MACURA vs M2AC** (truncation vs masking?).

## 1.3 The probabilistic ensemble

`models/ensemble.py` learns an ensemble of Gaussian MLPs, each predicting the
**next-state delta** as a Gaussian (paper Eq. 3). Where the members **agree**, the
model is confident (low epistemic uncertainty); where they **disagree** — usually
far from training data — it is not (Sec. 2.2). The plot trains a small *real*
ensemble on a 1-D function: members overlap inside the training region and fan
out beyond it. Look for the spread growing outside the grey band.

In [ ]:
plots.plot_ensemble_toy_1d(cfg_run['ensemble'], device=DEVICE,
                           save_path=f'{DRIVE}/plots/p1_ensemble_toy.png');

## 1.4 GJS uncertainty (paper Eq. 15–19)

MACURA measures epistemic uncertainty with the **geometric Jensen–Shannon (GJS)
divergence** averaged over all ensemble-member pairs (Eq. 15). Unlike the KL-based
measure, GJS has a **closed form for diagonal Gaussians** (Eq. 17–19), so it is
cheap to compute for every imagined transition — implemented as `compute_gjs` in
`algorithms/macura.py`. The plot shows GJS rising as the members' means spread
apart: that is exactly the 'high uncertainty' signal that will truncate rollouts.

In [ ]:
plots.plot_gjs_vs_spread(save_path=f'{DRIVE}/plots/p1_gjs_vs_spread.png');

## 1.5 Adaptive threshold κ and the rollout rule (Eq. 21)

The threshold **κ** is self-tuning (Eq. 21): the running mean of `ξ ×` (the
ζ-quantile of first-step uncertainties), with paper defaults `ζ=95%` and
`T_max=10` (Sec. 6.1) — leaving **ξ** as the single knob. The rollout rule
(`macura_rollout`): **continue while `u_GJS < κ`, else truncate** (Algorithm 2).
The cartoon shows GJS rising along one rollout and being cut at the κ crossing.

In [ ]:
plots.plot_rollout_truncation_cartoon(save_path=f'{DRIVE}/plots/p1_truncation.png');

## 1.6 UTD gradient scaling (Eq. 22)

Because adaptive rollouts produce a *variable* amount of imagined data, MACURA
scales the number of SAC gradient steps with how full the model buffer is
(Eq. 22), implemented as `gradient_steps` in `algorithms/macura.py`. This
update-to-data (UTD) ratio strongly affects model-free stability, so it is also a
**confound** we must control when comparing algorithms (see Part 3 and the
fairness audit in Part 6).

## 1.7 Imports & sanity

Import every project module, print library versions, and show the resolved config
so the exact hyperparameters used downstream are visible.

In [ ]:
import numpy, gymnasium, mujoco, stable_baselines3
from envs import drone_env
from models import ensemble
from algorithms import macura, mbpo, m2ac, sac
from training import train as trainer
print('torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| mujoco', mujoco.__version__, '| gymnasium', gymnasium.__version__,
      '| numpy', numpy.__version__)
import json; print('\nResolved config:'); print(json.dumps(cfg_run, indent=2)[:1500], '...')

In [ ]:
# env created WITHOUT a renderer — building the MuJoCo Renderer is the step that
# can crash the kernel, so it is isolated to the optional cell below + videos.
env, obs_dim, act_dim = drone_env.make_env(cfg_run['env'], seed=0, render=False)
print(f'obs_dim={obs_dim}, act_dim={act_dim}')
print('actuator ctrl ranges:\n', env.model.actuator_ctrlrange)

In [ ]:
# (Optional) still frame — only if RENDER is True. Creating the MuJoCo Renderer
# can crash the kernel on some GPUs; rendering is not needed for any result.
if RENDER:
    try:
        _renv, _, _ = drone_env.make_env(cfg_run['env'], seed=0, render=True)
        _renv.reset(seed=0)
        plots.render_frame(_renv, save_path=f'{DRIVE}/plots/p2_render.png')
        _renv.close()
    except Exception as e:
        print('rendering unavailable (skipping):', e)
else:
    print('RENDER=False -> skipping still-frame (not needed for results)')

## 2.1 Random policy → the platform tumbles fast

Random motor commands flip the drone within a fraction of a second. The tilt and
position-error traces blowing up is the instability that makes long imagined
rollouts dangerous.

In [ ]:
rec = plots.collect_rollout(env, policy='random', num_steps=250, seed=0)
plots.plot_rollout_traces(rec, title='Random policy (unstable)',
                          save_path=f'{DRIVE}/plots/p2_random.png');

## 2.2 Free dynamics (constant zero action)

With no corrective control the drone simply falls — a baseline for what the policy
must overcome.

In [ ]:
rec0 = plots.collect_rollout(env, policy='hover', num_steps=250, seed=0)
plots.plot_rollout_traces(rec0, title='Constant zero action (free dynamics)',
                          save_path=f'{DRIVE}/plots/p2_free.png');

## 2.3 State distribution under a random policy

The spread of altitude, tilt, and position error the ensemble must learn to model.

In [ ]:
plots.plot_state_distributions(env, num_episodes=15, num_steps=80,
                               save_path=f'{DRIVE}/plots/p2_state_dist.png');

## 2.4 Reward landscape (1-D and 2-D)

The dense quadratic reward is **smooth and peaked at the target** — exactly what
the ensemble can predict accurately (a sparse/discontinuous reward would corrupt
the uncertainty estimate). The 2-D heatmap shows the smooth basin over position.

In [ ]:
plots.plot_reward_landscape(cfg_run, save_path=f'{DRIVE}/plots/p2_reward_1d.png')
plots.plot_reward_heatmap_2d(cfg_run, save_path=f'{DRIVE}/plots/p2_reward_2d.png');

## 2.5 Action → state response (sanity)

Sweeping the thrust command should monotonically change altitude — confirming the
action interface behaves physically before we trust any learning curve.

In [ ]:
plots.plot_action_response(env, save_path=f'{DRIVE}/plots/p2_action_response.png');

---
# Part 3 — Paper vs project: design choices, explained

For each choice: *what the paper does*, *what this project does*, *why*, and a
*caveat*. These are the decisions that determine whether the comparison is fair.

## 3.1 Exploration noise — pink vs white

**Pink noise** is `1/f`, temporally *correlated* (smooth, momentum-like
exploration); **white noise** is uncorrelated (jittery).

- **Paper:** uses **pink-noise exploration for MACURA** but interacts
  *deterministically* for MBPO/M2AC (Sec. 6.2 / 7.3; Eberhard et al. 2023).
- **This project:** applies **one** scheme to **all four** (`exploration.type` in
  the config, `_explore` in `training/train.py`).
- **Why:** giving only MACURA a better explorer would conflate *exploration* with
  *rollout adaptation* — an unfair advantage (`tasks.md` Phase 4).
- **Caveat:** our pink generator is a simple FFT 1/f filter; the per-step noise in
  the loop is currently an AR(1) proxy (a Phase-4 TODO to make it exact).

The plot shows pink vs white as a time series and their power spectra — pink has
the characteristic `1/f` downward slope.

In [ ]:
plots.plot_noise_comparison(save_path=f'{DRIVE}/plots/p3_noise.png');

## 3.2 Rollout strategy

- **MACURA:** length set *per rollout* by κ vs GJS (adaptive truncation).
- **MBPO:** fixed **truncated-linear** schedule (length ramps with training step).
- **M2AC:** fixed length, then **mask** the least-trustworthy transitions.
- **SAC:** no model rollouts.

Schematic of rollout length vs step for each (the real MACURA curve appears in
Part 5.4). **Caveat:** MBPO's schedule must be *tuned* on the drone or it becomes
a strawman (`tasks.md` Phase 4).

In [ ]:
plots.plot_rollout_strategy_schematic(save_path=f'{DRIVE}/plots/p3_strategy.png');

## 3.3 Reward & termination

- **Paper:** theory assumes a **known** reward `r(s,a)` (Sec. 4); mbrl-lib often
  *learns* it in practice.
- **This project:** uses a single **known analytic** dense reward and termination
  (`known_reward_fn` / `termination_fn` in `envs/drone_env.py`), shared by all
  four so imagined transitions are scored identically.
- **Why:** removes a confound and keeps the GJS signal about *dynamics* error only.
- **Caveat:** model rollouts must respect the same crash envelope or the ensemble
  hallucinates flight past a crash — verified in the smoke test (Part 4.1).

## 3.4 Backbone — mbrl-lib vs Stable-Baselines3

- **Paper:** builds on **mbrl-lib** (Pineda 2021).
- **This project:** uses **Stable-Baselines3** SAC + our own PyTorch ensemble.
- **Why:** mbrl-lib is unmaintained and **not installable on Colab's Python 3.12**
  (forces ancient `omegaconf`/`gym`, incompatible with numpy 2) — see `README.md`
  / `tasks.md`. SB3 is maintained and installs cleanly.
- **Caveat:** SB3 is driven manually for the Dyna loop; confirm its API on the
  installed version (`tasks.md` Phase 2). Fairness is preserved: all four share
  the same SB3 SAC and the same ensemble.

## 3.5 Domain & the hypothesis under test

- **Paper:** validates on MuJoCo **locomotion** (HalfCheetah/Hopper/Walker/Ant/
  Humanoid) — fairly stable systems (Sec. 7).
- **This project:** an **unstable quadrotor** where errors compound fast.
- **Hypothesis the experiments test:** on an unstable system, *adaptive* truncation
  (MACURA) keeps model-based learning stable and sample-efficient where a *fixed*
  horizon (MBPO) over-imagines and wobbles/collapses — and beats model-free SAC on
  real-step efficiency. A mixed/negative result is still a valid finding.

---
# Part 4 — Running the comparison

Training runs on the GPU via `trainer.train_one(algo, cfg, DRIVE, seed)`, which
implements the Dyna loop and logs the curves. With `SMOKE=True` the runs are tiny.

## 4.1 Smoke test — one short run to validate the loop

Confirm a MACURA run logs curves, κ updates and stays finite, and rollouts respect
the termination envelope. Look for non-NaN κ and a printed eval return.

In [ ]:
# seed 99 so this sanity run never collides with the matrix seeds in 4.2
smoke_run = trainer.train_one('macura', make_cfg(True), DRIVE, seed=99)
ks = [k for _, k in smoke_run['kappa']]
import numpy as np
print('kappa finite:', bool(np.all(np.isfinite(ks))) if ks else 'n/a',
      '| eval returns:', [round(r, 1) for r in smoke_run['eval_return']])

## 4.2 Full matrix — 4 algorithms × seeds (idempotent)

Loops over algorithms and seeds, **skipping any run whose log JSON already exists**
in Drive (so a disconnected session resumes cheaply). Under `SMOKE=True` this is
fast; with `SMOKE=False` it is the real (long) experiment. Results are loaded back
into `runs` for plotting.

In [ ]:
import os, json
runs = []
for algo in cfg_run['experiment']['algorithms']:
    for seed in cfg_run['experiment']['seeds']:
        log_path = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(log_path):
            print('skip (exists):', algo, seed)
            runs.append(json.load(open(log_path)))
        else:
            runs.append(trainer.train_one(algo, cfg_run, DRIVE, seed=seed))
print('collected', len(runs), 'runs')

---
# Part 5 — Results & figures

Each figure states the claim it tests. **Expected ordering:** MACURA ≳ M2AC ≳ MBPO
> SAC early on, with MBPO at risk of wobbling/collapsing on the unstable drone.
(With `SMOKE=True` the curves are too short to show this — they validate the
plumbing; run `SMOKE=False` for the real story.)

## 5.1 Sample efficiency — return vs real env steps
**Claim:** model-based methods (esp. MACURA) reach a given return in fewer real
steps than SAC.

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/p5_sample_eff.png');

## 5.2 Failure rate while learning
**Claim:** adaptivity reduces crashes during training (the safety/cost story).

In [ ]:
plots.plot_failure_rate(runs, save_path=f'{DRIVE}/plots/p5_failure.png');

## 5.3 Final-policy quality
**Claim:** best achieved return per algorithm (hover quality).

In [ ]:
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/p5_final_quality.png');

## 5.4 MACURA signature — rollout length & κ
**Claim (the headline):** rollout length grows as the model becomes reliable while
κ falls (paper Fig. 8 analog).

In [ ]:
mac = next((r for r in runs if r['algo'] == 'macura'), None)
if mac: plots.plot_rollout_depth(mac, save_path=f'{DRIVE}/plots/p5_rollout_depth.png');

## 5.5 Model uncertainty over training
**Claim:** first-step GJS decreases as the ensemble learns the dynamics.

In [ ]:
if mac: plots.plot_gjs_over_training(mac, save_path=f'{DRIVE}/plots/p5_gjs.png');

## 5.6 Rollout-length distribution — early vs late
**Claim:** adaptivity made visible — short rollouts early (uncertain), longer late.

In [ ]:
if mac: plots.plot_rollout_length_hist(mac, save_path=f'{DRIVE}/plots/p5_len_hist.png');

## 5.7 Sample efficiency to a target return
**Claim:** real steps to first reach a fixed return — the 'does the model help?'
number. (Pick a threshold sensible for your run length.)

In [ ]:
import numpy as np
all_ret = [v for r in runs for v in r['eval_return']]
target = float(np.percentile(all_ret, 60)) if all_ret else 0.0
plots.plot_steps_to_target(runs, target, save_path=f'{DRIVE}/plots/p5_steps_to_target.png');

## 5.8 Per-seed spaghetti + mean
**Claim:** honest variance — individual seeds faint behind the mean.

In [ ]:
plots.plot_sample_efficiency_spaghetti(runs, save_path=f'{DRIVE}/plots/p5_spaghetti.png');

## 5.9 Return vs wall-clock
**Claim / caveat (README §7):** MBRL is sample-efficient but compute-heavy —
compute, not samples, is the Colab bottleneck.

In [ ]:
plots.plot_return_vs_wallclock(runs, save_path=f'{DRIVE}/plots/p5_wallclock.png');

## 5.10 (Optional) MuJoCo sanity anchor — Hopper
If time allows, running the *same* algorithm code on a stock MuJoCo task and
checking the curve reproduces the paper's qualitative result proves the
implementation and de-risks attributing drone failures to bugs (`tasks.md` Phase
6). This requires a small `make_env` variant for Gymnasium-MuJoCo and is left as a
guided TODO (do not block the drone study on it).

---
# Part 5b — Record & download flight videos

Load each trained agent's checkpoint, fly one deterministic evaluation episode
while capturing Skydio X2 frames, encode to mp4, save to Drive, display inline,
and offer a download. The recording logic is the pure helper
`viz.plots.record_policy_video`; the notebook only calls it.

In [ ]:
from stable_baselines3 import SAC
video_paths = {}
if not RENDER:
    print('RENDER=False -> skipping video recording. Set RENDER=True (hardening cell) to enable.')
else:
    render_env, _, _ = drone_env.make_env(cfg_run['env'], seed=999, render=True)
    for algo in cfg_run['experiment']['algorithms']:
        seed = cfg_run['experiment']['seeds'][0]
        ckpt = f'{DRIVE}/checkpoints/{algo}_seed{seed}.zip'
        if not os.path.exists(ckpt):
            print('no checkpoint for', algo, '- run Part 4 first'); continue
        agent = SAC.load(ckpt, device=DEVICE)
        out = f'{DRIVE}/videos/{algo}_seed{seed}.mp4'
        plots.record_policy_video(agent, render_env, num_steps=cfg_run['env']['max_episode_steps'],
                                  save_path=out, fps=30, also_gif=True)
        video_paths[algo] = out
        print('saved', out)
    render_env.close()

In [ ]:
# display inline
from IPython.display import Video, display
for algo, path in video_paths.items():
    print(algo.upper(), '->', path)
    display(Video(path, embed=True, width=480))

In [ ]:
# optional: download each video locally
from google.colab import files
for path in video_paths.values():
    files.download(path)

---
# Part 6 — Discussion & verification

Read the results back against the Part-3 hypotheses, audit fairness, and print a
reproducibility footer.

## 6.1 Results vs hypotheses

Answer honestly using the figures above:
- **Did the model help?** MACURA/MBPO/M2AC vs SAC on sample efficiency (5.1, 5.7).
- **Did adaptivity matter?** MACURA vs MBPO on stability/failures (5.1, 5.2, 5.4).
- **Truncation vs masking?** MACURA vs M2AC (5.1, 5.3).

A mixed/negative result on the unstable drone is a valid finding — note it rather
than overclaiming. The summary table below prints the best return per run.

In [ ]:
import pandas as pd
rows = [{'algo': r['algo'], 'seed': r['seed'],
         'best_return': round(max(r['eval_return']), 1) if r['eval_return'] else None,
         'final_fail_rate': round(r['eval_failure_rate'][-1], 2) if r['eval_failure_rate'] else None}
        for r in runs]
display(pd.DataFrame(rows).sort_values(['algo', 'seed']))

## 6.2 Fairness audit (`tasks.md` Phase 4)

Confirm the controls held: **one** exploration scheme for all four; κ / GJS finite
(no covariance blow-up); MBPO schedule is a real (tunable) schedule, not a
strawman; UTD policy documented. State any deviation explicitly.

In [ ]:
import numpy as np
print('exploration scheme (all algos):', cfg_run['exploration']['type'])
print('MACURA adaptive UTD:', cfg_run['rollout']['macura']['adaptive_gradient_steps'],
      '| MBPO/M2AC fixed UTD:', cfg_run['rollout']['mbpo']['fixed_gradient_steps'])
print('MBPO schedule [min,max,start,end]:', cfg_run['rollout']['mbpo']['rollout_schedule'])
if mac and mac['kappa']:
    kv = [k for _, k in mac['kappa']]
    print('kappa finite:', bool(np.all(np.isfinite(kv))), '| kappa range:',
          round(min(kv), 3), '->', round(max(kv), 3))
if mac and mac['gjs']:
    gv = [g for _, g in mac['gjs']]
    print('GJS finite:', bool(np.all(np.isfinite(gv))))

## 6.3 Reproducibility footer + what to run next

Print seeds, library versions, the config, and every artifact produced, then the
next steps tied to `tasks.md`.

In [ ]:
import os, hashlib, json
cfg_hash = hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:8]
print('SMOKE:', SMOKE, '| seeds:', cfg_run['experiment']['seeds'],
      '| total_env_steps:', cfg_run['experiment']['total_env_steps'])
print('config md5:', cfg_hash)
print('versions:', 'torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| mujoco', mujoco.__version__, '| gymnasium', gymnasium.__version__,
      '| numpy', numpy.__version__)
for sub in ['plots', 'videos', 'logs', 'checkpoints']:
    d = f'{DRIVE}/{sub}'
    if os.path.isdir(d):
        print(f'\n{sub}:'); [print('  ', f) for f in sorted(os.listdir(d))]
print('\nWhat to run next (tasks.md):')
print('  - set SMOKE=False and run the full 4x{3,5}-seed matrix to total_env_steps')
print('  - Phase 2: confirm SB3 _setup_learn / replay_buffer.add on installed version')
print('  - Phase 4: tune MBPO schedule + MACURA xi; exact pink-noise generator')
print('  - Phase 5/6: MuJoCo Hopper sanity anchor; robustness with process noise')